In [9]:
import tensorflow as tf
import os
import shutil
import random

# Defining Dataset Paths
base_dir = '/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog'  # Corrected dataset path
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# Creating train and validation directories
os.makedirs(os.path.join(train_dir, 'Cat'), exist_ok=True)
os.makedirs(os.path.join(train_dir, 'Dog'), exist_ok=True)
os.makedirs(os.path.join(validation_dir, 'Cat'), exist_ok=True)
os.makedirs(os.path.join(validation_dir, 'Dog'), exist_ok=True)

# Original dataset folders
cat_dir = os.path.join(base_dir, 'Cat')  # Adjusted to match the folder structure
dog_dir = os.path.join(base_dir, 'Dog')

# Function to split data with progress tracking
def split_data(source_dir, train_dir, validation_dir, split_size=0.8):
    files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]
    random.shuffle(files)
    split_index = int(len(files) * split_size)
    train_files = files[:split_index]
    validation_files = files[split_index:]

    total_files = len(files)
    processed = 0

    for file in train_files:
        shutil.move(os.path.join(source_dir, file), os.path.join(train_dir, file))
        processed += 1
        if processed % 1000 == 0:
            print(f"Moved {processed}/{total_files} files to training set...")

    for file in validation_files:
        shutil.move(os.path.join(source_dir, file), os.path.join(validation_dir, file))
        processed += 1
        if processed % 1000 == 0:
            print(f"Moved {processed}/{total_files} files to validation set...")

    print(f"Splitting complete. Total files processed: {processed}")

# Split cat and dog images
split_data(cat_dir, os.path.join(train_dir, 'Cat'), os.path.join(validation_dir, 'Cat'))
split_data(dog_dir, os.path.join(train_dir, 'Dog'), os.path.join(validation_dir, 'Dog'))

print("Dataset successfully split into training and validation sets.")


Moved 1000/2500 files to training set...
Moved 2000/2500 files to training set...
Splitting complete. Total files processed: 2500
Moved 1000/2500 files to training set...
Moved 2000/2500 files to training set...
Splitting complete. Total files processed: 2500
Dataset successfully split into training and validation sets.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Defining dataset directories
base_dir = '/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# Define data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# No augmentation for validation, only normalization
valid_datagen = ImageDataGenerator(rescale=1./255)

# Loading and preprocess the data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'  # Using 'binary' for 2-class classification
)

validation_generator = valid_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

# Checking if data is loading correctly
print(f"Training batches: {len(train_generator)}, Validation batches: {len(validation_generator)}")


Found 4000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Training batches: 125, Validation batches: 32


In [5]:
# Defining the CNN Model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compiling the Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Training the Model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=validation_generator,
    validation_steps=len(validation_generator),
    epochs=20,
    verbose=1
)

# Save the Model
model.save('/content/drive/MyDrive/For Collab/Quiz_and_assignments/cats_and_dogs/cat_dog_classifier.keras')

print("Model training complete and saved successfully!")


Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 56s 429ms/step - accuracy: 0.4968 - loss: 0.8908 - val_accuracy: 0.5000 - val_loss: 0.6929
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 76s 395ms/step - accuracy: 0.5089 - loss: 0.6931 - val_accuracy: 0.5260 - val_loss: 0.6852
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 81s 387ms/step - accuracy: 0.5048 - loss: 0.6940 - val_accuracy: 0.6250 - val_loss: 0.6911
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 80s 371ms/step - accuracy: 0.5408 - loss: 0.6911 - val_accuracy: 0.5210 - val_loss: 0.6842
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 46s 368ms/step - accuracy: 0.5362 - loss: 0.6862 - val_accuracy: 0.6240 - val_loss: 0.6863
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 81s 359ms/step - accuracy: 0.5456 - loss: 0.6855 - val_accuracy: 0.6720 - val_loss: 0.6669
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 45s 357ms/step - accuracy: 0.5634 - loss: 0.6794 - val_accuracy: 0.5960 - val_loss: 0.6670
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 360ms/step - accuracy: 0.5962 - loss: 0

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/For Collab/Quiz_and_assignments/cats_and_dogs/cat_dog_classifier.keras'

In [6]:
# Creating the directory so that the model can be saved.
import os

save_dir = '/content/drive/MyDrive/For Collab/Quiz_and_assignments/cats_and_dogs/'
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, 'cat_dog_classifier.keras'))


In [7]:
#Prediction (Loading the model)
from tensorflow import keras

# Load the saved model
model = keras.models.load_model('/content/drive/MyDrive/For Collab/Quiz_and_assignments/cats_and_dogs/cat_dog_classifier.keras')


In [8]:
#preprocessing function
import numpy as np
from tensorflow.keras.preprocessing import image

def preprocess_image(img_path, img_size=(150, 150)):
    img = image.load_img(img_path, target_size=img_size)  # Resize image
    img_array = image.img_to_array(img)  # Convert to array
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize to [0,1] range
    return img_array


In [9]:
# Prediction function
def predict_image(img_path):
    img_array = preprocess_image(img_path)
    prediction = model.predict(img_array)  # Get model's output
    class_label = "Dog" if prediction[0] > 0.5 else "Cat"  # Assuming binary classification
    print(f"Prediction: {class_label} ({prediction[0][0]:.2f})")


In [10]:
# Checking prediction for a specific image
test_img = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog/sample_1.jpg"  # Replace with actual path
predict_image(test_img)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 778ms/step
Prediction: Dog (0.50)
